In [ ]:
import json
import time
import requests
from bs4 import BeautifulSoup

def get_perfect_filter():
    """Generates a strict filter that ONLY returns exactly the fields we need."""
    print("Generating strict API filter...")
    url = "https://api.stackexchange.com/2.3/filters/create"
    
    # base='none' means we start with a blank slate, saving massive bandwidth
    params = {
        "base": "none",
        "include": ".wrapper.items;.wrapper.has_more;.wrapper.backoff;"
                   "question.body;question.answers;question.answer_count;"
                   "answer.body;answer.score"
    }
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        filter_id = response.json().get("filter")
        print(f"Success! Filter ID: {filter_id}")
        return filter_id
    else:
        raise Exception("Failed to generate filter. Check internet connection.")

def fetch_data_direct(topics, pages_per_topic=90, output_filename="so_data.json", api_key=None):
    my_filter = get_perfect_filter()
    all_pairs = []

    for topic in topics:
        print(f"\n--- Fetching topic: {topic} ---")
        
        for page in range(1, pages_per_topic + 1):
            url = "https://api.stackexchange.com/2.3/questions"
            params = {
                "page": page,
                "pagesize": 100,      # Max items per page
                "order": "desc",
                "sort": "votes",      # Get the highest quality questions first
                "tagged": topic,
                "site": "stackoverflow",
                "filter": my_filter   # Our perfectly generated filter
            }
            
            if api_key:
                params['key'] = api_key

            # 1. Make the raw request
            res = requests.get(url, params=params)
            data = res.json()

            # 2. Catch actual API errors immediately
            if 'error_id' in data:
                print(f"FATAL API ERROR: {data.get('error_message')}")
                return

            items = data.get('items', [])
            if not items:
                print(f"No more items found for {topic} on page {page}. Moving to next topic.")
                break

            # 3. Parse the items
            for q in items:
                if q.get('answer_count', 0) > 0 and 'answers' in q:
                    # Find the answer with the highest score
                    top_answer = max(q['answers'], key=lambda x: x.get('score', 0))
                    
                    raw_q = q.get('body', '')
                    raw_a = top_answer.get('body', '')
                    
                    clean_q = BeautifulSoup(raw_q, "html.parser").get_text(separator='\n').strip()
                    clean_a = BeautifulSoup(raw_a, "html.parser").get_text(separator='\n').strip()
                    
                    all_pairs.append({
                        "topic": topic,
                        "question_body": clean_q,
                        "answer_body": clean_a
                    })

            print(f"[{topic}] Page {page}/{pages_per_topic} parsed. Total pairs collected: {len(all_pairs)}")

            # 4. Mandatory API etiquette (Prevents IP Bans)
            if 'backoff' in data:
                sleep_time = data['backoff']
                print(f"API requested a backoff. Sleeping for {sleep_time} seconds...")
                time.sleep(sleep_time)
            else:
                time.sleep(1.5) # Polite delay between requests
                
            # If the API says there are no more pages, stop looping
            if not data.get('has_more', False):
                break

    # 5. Save the data
    print(f"\nFinished! Saving {len(all_pairs)} Q&A pairs to {output_filename}...")
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(all_pairs, f, indent=4, ensure_ascii=False)

# ==========================================
# Example Usage:
# ==========================================
topics_list = ["python", "javascript", "java", "c#", "cpp", "android", "html", "jquery", "php", "ios", "sql", "css"]

# Test with 1 page per topic first!
fetch_data_direct(
    topics=topics_list, 
    pages_per_topic=1, 
    output_filename="training_data.json",
    api_key="rl_gKxM4cKc7fHnvEKQWXgG4tSRA"
)

In [ ]:
import json
import requests

def debug_stackoverflow_api():
    print("1. Generating a fresh filter (base='default' + bodies/answers)...")
    url_filter = "https://api.stackexchange.com/2.3/filters/create"
    
    # We will use the default base this time, just to be absolutely sure 
    # we aren't accidentally excluding a mandatory hidden field.
    params_filter = {
        "base": "default", 
        "include": "question.body;question.answers;answer.body"
    }
    
    res_filter = requests.get(url_filter, params=params_filter)
    filter_id = res_filter.json().get("filter")
    print(f"   Success! Filter ID: {filter_id}")

    print("\n2. Fetching 5 top Python questions...")
    url_qa = "https://api.stackexchange.com/2.3/questions"
    params_qa = {
        "page": 1,
        "pagesize": 5, # Just 5 items for a quick debug
        "order": "desc",
        "sort": "votes",
        "tagged": "python",
        "site": "stackoverflow",
        "filter": filter_id
    }
    
    res_qa = requests.get(url_qa, params=params_qa)
    data = res_qa.json()
    
    if 'error_id' in data:
        print(f"\nFATAL API ERROR: {data}")
        return
        
    items = data.get('items', [])
    print(f"\n3. Received {len(items)} questions from the API.")
    
    if len(items) > 0:
        first_q = items[0]
        
        print("\n==================================================")
        print("   DEBUG: RAW JSON OF THE FIRST QUESTION")
        print("==================================================")
        # Print the exact keys the API returned for the first question
        print(json.dumps(list(first_q.keys()), indent=4))
        
        print("\n==================================================")
        print("   DEBUG: CHECKING FOR TARGET DATA")
        print("==================================================")
        
        if 'body' in first_q:
            print("[✓] Question Body found!")
        else:
            print("[X] Question Body MISSING.")
            
        if 'answers' in first_q:
            print(f"[✓] Answers array found! Contains {len(first_q['answers'])} answers.")
            if 'body' in first_q['answers'][0]:
                print("[✓] Top Answer Body found!")
            else:
                print("[X] Top Answer Body MISSING.")
        else:
            print("[X] Answers array completely MISSING.")

# Run the debug test
debug_stackoverflow_api()

In [ ]:
import json
import time
import requests
from bs4 import BeautifulSoup

def get_perfect_filter():
    """Generates a strict filter that ONLY returns exactly the fields we need."""
    print("Generating strict API filter...")
    url = "https://api.stackexchange.com/2.3/filters/create"
    
    params = {
        "base": "none",
        "include": ".wrapper.items;.wrapper.has_more;.wrapper.backoff;"
                   "question.body;question.answers;question.answer_count;"
                   "answer.body;answer.score"
    }
    response = requests.get(url, params=params)
    data = response.json()
    
    # THE FIX: We must look inside the 'items' array to get the filter string!
    if 'items' in data and len(data['items']) > 0:
        filter_id = data['items'][0]['filter']
        print(f"Success! Filter ID extracted: {filter_id}")
        return filter_id
    else:
        raise Exception(f"Failed to generate filter. API returned: {data}")

def fetch_data_direct(topics, pages_per_topic=90, output_filename="so_data.json", api_key=None):
    my_filter = get_perfect_filter()
    all_pairs = []

    for topic in topics:
        print(f"\n--- Fetching topic: {topic} ---")
        
        for page in range(1, pages_per_topic + 1):
            url = "https://api.stackexchange.com/2.3/questions"
            params = {
                "page": page,
                "pagesize": 100,      
                "order": "desc",
                "sort": "votes",      
                "tagged": topic,
                "site": "stackoverflow",
                "filter": my_filter   
            }
            
            if api_key:
                params['key'] = api_key

            res = requests.get(url, params=params)
            data = res.json()

            if 'error_id' in data:
                print(f"FATAL API ERROR: {data.get('error_message')}")
                return

            items = data.get('items', [])
            if not items:
                print(f"No more items found for {topic} on page {page}. Moving to next topic.")
                break

            for q in items:
                if q.get('answer_count', 0) > 0 and 'answers' in q:
                    top_answer = max(q['answers'], key=lambda x: x.get('score', 0))
                    
                    raw_q = q.get('body', '')
                    raw_a = top_answer.get('body', '')
                    
                    clean_q = BeautifulSoup(raw_q, "html.parser").get_text(separator='\n').strip()
                    clean_a = BeautifulSoup(raw_a, "html.parser").get_text(separator='\n').strip()
                    
                    all_pairs.append({
                        "topic": topic,
                        "question_body": clean_q,
                        "answer_body": clean_a
                    })

            print(f"[{topic}] Page {page}/{pages_per_topic} parsed. Total pairs collected: {len(all_pairs)}")

            if 'backoff' in data:
                sleep_time = data['backoff']
                print(f"API requested a backoff. Sleeping for {sleep_time} seconds...")
                time.sleep(sleep_time)
            else:
                time.sleep(1.5) 
                
            if not data.get('has_more', False):
                break

    print(f"\nFinished! Saving {len(all_pairs)} Q&A pairs to {output_filename}...")
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(all_pairs, f, indent=4, ensure_ascii=False)

# ==========================================
# Example Usage:
# ==========================================
topics_list = ["python", "javascript", "java", "c#", "cpp", "android", "html", "jquery", "php", "ios", "sql", "css"]

# Run it! It will now actually print a real string like '!*SU8CGYZ...' 
# and the bodies will finally pull correctly.
fetch_data_direct(
    topics=topics_list, 
    pages_per_topic=1, 
    output_filename="training_data.json",
    api_key="rl_gKxM4cKc7fHnvEKQWXgG4tSRA"
)